# Heart Disease Prediction – EDA, Risk Factors, and Model Comparison

This notebook analyzes the **Indicators of Heart Disease (2022 UPDATE)** dataset
(“Personal Key Indicators of Heart Disease”). It:

1. Performs Exploratory Data Analysis (EDA)
2. Tests **H1 (Primary – Risk Factors)**:
   > Age, BMI, diabetes, smoking, physical inactivity, poor general health,
   > and inadequate sleep are positively associated with self-reported heart disease
   > and rank among the most important predictors.
3. Tests **H2 (Secondary – Model Performance)**:
   > Non-linear ML models (Random Forest, XGBoost) achieve higher AUROC than
   > a baseline Logistic Regression model on the same predictors.
4. Builds and tunes models with stratified cross-validation
5. Evaluates models on a held-out test set
6. Provides explainability (coefficients, feature importance, SHAP)
7. Conducts subgroup analyses (e.g., by age group, sex, race, diabetes)

Reproducibility is handled with the **`seedhash`** package, generating **16-bit seeds**
for all random_state parameters.

In [ ]:
# %% 
# --- 0. Setup: Imports, plotting style, and reproducible seeds ---

# Core Python utilities
import os
import random
import warnings

# Data science stack
import numpy as np
import pandas as pd

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning & preprocessing
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    RandomizedSearchCV
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Metrics
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Statistical tests
from scipy import stats

# Explainability
import shap

# Inference-focused logistic regression (for odds ratios etc.)
import statsmodels.api as sm

# Deterministic seed generation
from seedhash import SeedHashGenerator  # pip install seedhash

warnings.filterwarnings("ignore")

# --- 0.1. Global plotting style (feel free to customize) ---
sns.set(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

# --- 0.2. Reproducible seeds using seedhash (16-bit only) ---

def to_16bit_seed(seed_int: int) -> int:
    """
    Convert an arbitrary integer seed into a 16-bit seed [0, 65535].
    This enforces the constraint that we only use 16-bit seeds.
    """
    return int(seed_int) % (2**16)

# Create a deterministic seed generator keyed to this project
seed_gen = SeedHashGenerator("heart_disease_ml_project_v1")

# Generate a small pool of seeds and map them to specific uses
raw_seeds = seed_gen.generate_seeds(10)  # returns a list of ints

SEEDS = {
    "global":      to_16bit_seed(raw_seeds[0]),
    "train_test":  to_16bit_seed(raw_seeds[1]),
    "cv":          to_16bit_seed(raw_seeds[2]),
    "log_reg":     to_16bit_seed(raw_seeds[3]),
    "rf":          to_16bit_seed(raw_seeds[4]),
    "xgb":         to_16bit_seed(raw_seeds[5]),
}

# Apply global seeds for Python and NumPy
random.seed(SEEDS["global"])
np.random.seed(SEEDS["global"])

SEEDS

In [ ]:
# --- 0.0. Global config: output paths for metrics and plots ---

import os  # if not already imported above

METRICS_DIR = r"your\local\path\metrics"
PLOTS_DIR   = r"your\local\path\plots"

os.makedirs(METRICS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

## 1. Load Data and Basic Structure

Dataset: **Indicators of Heart Disease (2022 UPDATE)**  
Column list (provided):

- HeartDisease (target)
- BMI
- Smoking
- AlcoholDrinking
- Stroke
- PhysicalHealth
- MentalHealth
- DiffWalking
- Sex
- AgeCategory
- Race
- Diabetic
- PhysicalActivity
- GenHealth
- SleepTime
- Asthma
- KidneyDisease
- SkinCancer

In [ ]:
# %%
# --- 1.1. Load dataset with APA-style metric exports ---

# DATA_PATH stays the same
DATA_PATH = r"your\local\path\heart_2020_cleaned.csv"

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
df_head = df.head()
display(df_head)

# -------------------------------------------------------------------------
# Save APA-style metrics/tables for Section 1.1
# -------------------------------------------------------------------------
section_id = "Section1_1"

# ---- Table 1: dataset shape ---------------------------------------------
TABLE1_TITLE = "Shape of the Indicators of Heart Disease (2022 UPDATE) dataset"

shape_df = pd.DataFrame({
    "Metric": ["Number of observations", "Number of variables"],
    "Value": [df.shape[0], df.shape[1]],
})

table1_filename = f"{section_id}_Table1_Dataset_Shape.csv"
table1_path = os.path.join(METRICS_DIR, table1_filename)
shape_df.to_csv(table1_path, index=False)

print(f"\n[INFO] Saved Section 1.1 Table 1 (dataset shape) to:")
print(table1_path)
print(f"[INFO] Suggested APA label: Table X. {TABLE1_TITLE}.")


# ---- Table 2: first 5 rows ---------------------------------------------
TABLE2_TITLE = "First five rows of the Indicators of Heart Disease (2022 UPDATE) dataset"

table2_filename = f"{section_id}_Table2_First5_Rows_Dataset.csv"
table2_path = os.path.join(METRICS_DIR, table2_filename)
df_head.to_csv(table2_path, index=False)

print(f"\n[INFO] Saved Section 1.1 Table 2 (first five rows) to:")
print(table2_path)
print(f"[INFO] Suggested APA label: Table Y. {TABLE2_TITLE}.")

In [ ]:
# %%
# --- 1.2. Basic info and data types with APA-style metric exports ---

section_id = "Section1_2"

# -------------------------------------------------------------------------
# 1) Data info-style summary (like df.info) as an APA-ready table
#    File: Section1_2_Table1_DataInfo_NonMissing_Dtypes.csv
# -------------------------------------------------------------------------
TABLE1_TITLE = "Non-missing counts and data types for all variables"

info_df = pd.DataFrame({
    "Variable name": df.columns,
    "Non-missing n": df.notnull().sum().values,
    "Data type": df.dtypes.astype(str).values
})

print("DataFrame info (non-missing counts and data types):")
display(info_df)

table1_filename = f"{section_id}_Table1_DataInfo_NonMissing_Dtypes.csv"
table1_path = os.path.join(METRICS_DIR, table1_filename)
info_df.to_csv(table1_path, index=False)

print(f"\n[INFO] Saved Section 1.2 Table 1 (data info) to:")
print(table1_path)
print(f"[INFO] Suggested APA label: Table X. {TABLE1_TITLE}.")

# (Optional) still show raw df.info() console output
print("\nRaw df.info() output:")
df.info()

# -------------------------------------------------------------------------
# 2) Summary statistics for numeric columns
#    File: Section1_2_Table2_SummaryStats_Numeric.csv
# -------------------------------------------------------------------------
TABLE2_TITLE = "Descriptive statistics for numeric variables"

summary_stats = df.describe()  # numeric columns only by default

print("\nSummary statistics for numeric columns:")
display(summary_stats)

# Optionally give APA-style column labels (rename describe() columns)
summary_stats_apa = summary_stats.rename(
    columns={
        "count": "N",
        "mean": "Mean",
        "std": "Std. deviation",
        "min": "Minimum",
        "25%": "25th percentile",
        "50%": "Median",
        "75%": "75th percentile",
        "max": "Maximum",
    }
)

table2_filename = f"{section_id}_Table2_SummaryStats_Numeric.csv"
table2_path = os.path.join(METRICS_DIR, table2_filename)
summary_stats_apa.to_csv(table2_path)

print(f"\n[INFO] Saved Section 1.2 Table 2 (numeric summary stats) to:")
print(table2_path)
print(f"[INFO] Suggested APA label: Table Y. {TABLE2_TITLE}.")

# -------------------------------------------------------------------------
# 3) Unique values per column
#    File: Section1_2_Table3_UniqueValues_PerColumn.csv
# -------------------------------------------------------------------------
TABLE3_TITLE = "Number of distinct values for each variable"

unique_counts = df.nunique(dropna=False)
unique_df = unique_counts.reset_index()
unique_df.columns = ["Variable name", "Number of distinct values"]

print("\nUnique values per column:")
display(unique_df)

# Also print in the original simple format, if you like:
for col in df.columns:
    print(f"{col}: {df[col].nunique()} unique values")

table3_filename = f"{section_id}_Table3_UniqueValues_PerColumn.csv"
table3_path = os.path.join(METRICS_DIR, table3_filename)
unique_df.to_csv(table3_path, index=False)

print(f"\n[INFO] Saved Section 1.2 Table 3 (unique values per column) to:")
print(table3_path)
print(f"[INFO] Suggested APA label: Table Z. {TABLE3_TITLE}.")

## 2. Target Distribution and Class Imbalance

Here we inspect the distribution of **HeartDisease** (Yes/No) to assess
class imbalance, which is important for model choice and evaluation.

In [ ]:
# --- 2.1. Target encoding (Yes/No -> 1/0) and distribution, with APA-style outputs ---

section_id = "Section2_1"

# APA-style metadata
TABLE_TITLE  = "Distribution of self-reported heart disease status"
FIGURE_TITLE = "Class distribution of self-reported heart disease status"

# Target encoding
df["HeartDisease_binary"] = df["HeartDisease"].map({"No": 0, "Yes": 1})
if df["HeartDisease_binary"].isna().any():
    df["HeartDisease_binary"] = df["HeartDisease"].astype(int)

target_counts = df["HeartDisease_binary"].value_counts().sort_index()
target_props  = df["HeartDisease_binary"].value_counts(normalize=True).sort_index()

print("HeartDisease (0 = No, 1 = Yes) counts:")
print(target_counts)
print("\nProportions:")
print(target_props)

# -------------------------------------------------------------------------
# APA-style table for target distribution
#   File: Section2_1_Table1_HeartDisease_Distribution.csv
# -------------------------------------------------------------------------
status_labels = target_counts.index.map({0: "No heart disease", 1: "Heart disease"})

target_dist_df = pd.DataFrame({
    "Heart disease status": status_labels,
    "n": target_counts.values,
    "Percentage": (target_props.values * 100).round(2),
})

table_filename = f"{section_id}_Table1_HeartDisease_Distribution.csv"
table_path = os.path.join(METRICS_DIR, table_filename)
target_dist_df.to_csv(table_path, index=False)

print(f"\n[INFO] Saved Section 2.1 table to:")
print(table_path)
print(f"[INFO] Suggested APA label: Table X. {TABLE_TITLE}.")

# -------------------------------------------------------------------------
# Bar plot of class distribution – APA figure
#   File: Section2_1_Plot1_HeartDisease_Class_Distribution.png
# -------------------------------------------------------------------------
fig, ax = plt.subplots()
sns.barplot(
    x=status_labels,
    y=target_counts.values,
    ax=ax
)

ax.set_xlabel("Heart disease status")
ax.set_ylabel("Count")
ax.set_title(f"{FIGURE_TITLE}")

plt.tight_layout()

figure_filename = f"{section_id}_Plot1_HeartDisease_Class_Distribution.png"
figure_path = os.path.join(PLOTS_DIR, figure_filename)
fig.savefig(figure_path, dpi=300, bbox_inches="tight")

print(f"[INFO] Saved Section 2.1 plot to:")
print(figure_path)

plt.show()

## 3. Exploratory Data Analysis (EDA)

We examine:

- Distributions of key numeric risk factors (BMI, PhysicalHealth, MentalHealth, SleepTime)
- Differences in these distributions by heart disease status
- Prevalence by key categorical risk factors (Smoking, Diabetic, PhysicalActivity, GenHealth etc.)

In [ ]:
# --- 3.1. Define feature groups (numeric vs categorical) with APA-style export ---

numeric_features = ["BMI", "PhysicalHealth", "MentalHealth", "SleepTime"]

categorical_features = [
    "Smoking",
    "AlcoholDrinking",
    "Stroke",
    "DiffWalking",
    "Sex",
    "AgeCategory",
    "Race",
    "Diabetic",
    "PhysicalActivity",
    "GenHealth",
    "Asthma",
    "KidneyDisease",
    "SkinCancer",
]

# Sanity check
missing_cols = [
    col for col in numeric_features + categorical_features + ["HeartDisease_binary"]
    if col not in df.columns
]
assert len(missing_cols) == 0, f"Missing columns in dataset: {missing_cols}"

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

# -------------------------------------------------------------------------
# APA-style table for predictor types
#   File: Section3_1_Table1_Numeric_Categorical_Predictors.csv
# -------------------------------------------------------------------------
section_id   = "Section3_1"
TABLE_TITLE  = "Summary of numeric and categorical predictors"

feature_info = []

for col in numeric_features:
    feature_info.append({
        "Feature name": col,
        "Feature type": "Numeric",
    })

for col in categorical_features:
    feature_info.append({
        "Feature name": col,
        "Feature type": "Categorical",
    })

features_df = pd.DataFrame(feature_info)

table_filename = f"{section_id}_Table1_Numeric_Categorical_Predictors.csv"
table_path = os.path.join(METRICS_DIR, table_filename)
features_df.to_csv(table_path, index=False)

print(f"\n[INFO] Saved Section 3.1 table to:")
print(table_path)
print(f"[INFO] Suggested APA label: Table X. {TABLE_TITLE}.")

In [ ]:
# --- 3.2. Numeric distributions by heart disease status with APA-style exports ---

section_id = "Section3_2"

# -------------------------------------------------------------------------
# 3.2 – Metrics table: summary stats for numeric features by heart disease status
#   File: Section3_2_Table1_Numeric_Distributions_by_HeartDisease.csv
# -------------------------------------------------------------------------
TABLE_TITLE = "Descriptive statistics for numeric risk factors by heart disease status"

status_map = {0: "No heart disease", 1: "Heart disease"}
metrics_rows = []

for col in numeric_features:
    for status_value, status_label in status_map.items():
        mask = df["HeartDisease_binary"] == status_value
        series = df.loc[mask, col].dropna()
        if series.empty:
            continue
        
        metrics_rows.append({
            "Variable name": col,
            "Heart disease status": status_label,
            "n": series.shape[0],
            "Mean": series.mean(),
            "Std. deviation": series.std(),
            "Median": series.median(),
            "25th percentile": series.quantile(0.25),
            "75th percentile": series.quantile(0.75),
            "Minimum": series.min(),
            "Maximum": series.max(),
        })

numeric_hd_stats_df = pd.DataFrame(metrics_rows)

table_filename = f"{section_id}_Table1_Numeric_Distributions_by_HeartDisease.csv"
table_path = os.path.join(METRICS_DIR, table_filename)
numeric_hd_stats_df.to_csv(table_path, index=False)

print(f"\n[INFO] Saved Section 3.2 Table 1 (numeric distributions by heart disease) to:")
print(table_path)
print(f"[INFO] Suggested APA label: Table X. {TABLE_TITLE}.")

# -------------------------------------------------------------------------
# 3.2 – Plots: KDE + boxplot per numeric feature
#   Files:
#     Section3_2_Plot1_<Feature>_Distribution_by_HeartDisease.png
#     Section3_2_Plot2_<Feature>_Distribution_by_HeartDisease.png
#     ...
# -------------------------------------------------------------------------
for i, col in enumerate(numeric_features, start=1):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f"Distribution of {col} by heart disease status", fontsize=14)
    
    # Histogram / KDE (left)
    sns.kdeplot(
        data=df,
        x=col,
        hue="HeartDisease_binary",
        ax=axes[0],
        common_norm=False,
        fill=True,
        alpha=0.4,
        legend=True
    )
    axes[0].set_title(f"{col} distribution")
    axes[0].set_xlabel(col)
    axes[0].set_ylabel("Density")

    # Boxplot by heart disease status (right)
    sns.boxplot(
        data=df,
        x="HeartDisease_binary",
        y=col,
        ax=axes[1]
    )
    axes[1].set_xticklabels(["No heart disease", "Heart disease"])
    axes[1].set_title(f"{col} by heart disease status")
    axes[1].set_xlabel("Heart disease status")
    axes[1].set_ylabel(col)
    
    plt.tight_layout()
    
    # Save figure with your Section3_2_Plot<i>_... naming convention
    safe_col_name = col.replace(" ", "")
    figure_filename = f"{section_id}_Plot{i}_{safe_col_name}_Distribution_by_HeartDisease.png"
    figure_path = os.path.join(PLOTS_DIR, figure_filename)
    fig.savefig(figure_path, dpi=300, bbox_inches="tight")
    
    print(f"[INFO] Saved Section 3.2 Plot {i} for {col} to:")
    print(figure_path)
    
    plt.show()


In [ ]:
# --- 3.3. Correlation matrix for numeric variables with APA-style exports ---

section_id = "Section3_3"
TABLE_TITLE  = "Correlation matrix for numeric variables and heart disease status"
FIGURE_TITLE = "Correlation heatmap for numeric variables and heart disease status"

# -------------------------------------------------------------------------
# 3.3 – Correlation matrix (metrics table)
# -------------------------------------------------------------------------
corr = df[numeric_features + ["HeartDisease_binary"]].corr()

print("Correlation matrix (numeric features + HeartDisease_binary):")
display(corr)

# Build an APA-friendly version: add a 'Variable name' column
corr_apa = corr.copy()
corr_apa.insert(0, "Variable name", corr_apa.index)

table_filename = f"{section_id}_Table1_Correlation_Numeric_HeartDisease.csv"
table_path = os.path.join(METRICS_DIR, table_filename)
corr_apa.to_csv(table_path, index=False)

print(f"\n[INFO] Saved Section 3.3 Table 1 (correlation matrix) to:")
print(table_path)
print(f"[INFO] Suggested APA label: Table X. {TABLE_TITLE}.")

# -------------------------------------------------------------------------
# 3.3 – Correlation heatmap (plot)
# -------------------------------------------------------------------------
plt.figure(figsize=(6, 5))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    square=True
)
plt.title(FIGURE_TITLE)
plt.xlabel("Variables")
plt.ylabel("Variables")
plt.tight_layout()

figure_filename = f"{section_id}_Plot1_Correlation_Heatmap_Numeric_HeartDisease.png"
figure_path = os.path.join(PLOTS_DIR, figure_filename)
plt.savefig(figure_path, dpi=300, bbox_inches="tight")

print(f"[INFO] Saved Section 3.3 Plot 1 (correlation heatmap) to:")
print(figure_path)

plt.show()

In [ ]:
# --- 3.4. Prevalence of heart disease by key categorical risk factors with APA-style exports ---

section_id = "Section3_4"
TABLE_TITLE  = "Prevalence of self-reported heart disease by categorical risk factors"

categorical_risk_factors = ["AgeCategory", "GenHealth", "Smoking", "Diabetic", "PhysicalActivity", "Sex", "Race"]

# We'll collect all prevalence metrics here for a single combined table
metrics_rows = []

def plot_hd_prevalence_by_category(col, plot_index, top_n=None):
    """
    Plot percentage of heart disease cases within each category of `col`
    and append metrics to a global list for APA-style export.
    """
    # Crosstab: rows = categories, columns = HeartDisease_binary
    ctab = pd.crosstab(df[col], df["HeartDisease_binary"], normalize="index") * 100
    ctab = ctab.rename(columns={0: "No heart disease (%)", 1: "Heart disease (%)"}) \
               .sort_values("Heart disease (%)", ascending=False)
    
    if top_n is not None:
        ctab = ctab.head(top_n)
    
    print(f"\nHeart disease prevalence by {col}:")
    display(ctab)
    
    # ---------------------------------------------------------------------
    # Append to metrics_rows (long format: one row per category/variable)
    # ---------------------------------------------------------------------
    for level, row in ctab.iterrows():
        metrics_rows.append({
            "Risk factor variable": col,
            "Category": level,
            "Heart disease prevalence (%)": row["Heart disease (%)"],
            "No heart disease prevalence (%)": row["No heart disease (%)"],
        })
    
    # ---------------------------------------------------------------------
    # Plot (bar chart of HD prevalence) and save with Section3_4_Plot<i>_...
    # ---------------------------------------------------------------------
    plt.figure(figsize=(8, 4))
    sns.barplot(
        x=ctab.index,
        y=ctab["Heart disease (%)"],
    )
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Self-reported heart disease prevalence (%)")
    plt.xlabel(col)
    plt.title(f"Prevalence of self-reported heart disease by {col}")
    plt.tight_layout()
    
    safe_col_name = col.replace(" ", "")
    figure_filename = f"{section_id}_Plot{plot_index}_{safe_col_name}_HeartDisease_Prevalence.png"
    figure_path = os.path.join(PLOTS_DIR, figure_filename)
    plt.savefig(figure_path, dpi=300, bbox_inches="tight")
    
    print(f"[INFO] Saved Section 3.4 Plot {plot_index} for {col} to:")
    print(figure_path)
    
    plt.show()

# Generate plots + metrics for each categorical risk factor
for i, cat in enumerate(categorical_risk_factors, start=1):
    plot_hd_prevalence_by_category(cat, plot_index=i)

# -------------------------------------------------------------------------
# Combined APA-style metrics table for Section 3.4
#   File: Section3_4_Table1_HeartDisease_Prevalence_by_Categorical_RiskFactors.csv
# -------------------------------------------------------------------------
prevalence_df = pd.DataFrame(metrics_rows)

table_filename = f"{section_id}_Table1_HeartDisease_Prevalence_by_Categorical_RiskFactors.csv"
table_path = os.path.join(METRICS_DIR, table_filename)
prevalence_df.to_csv(table_path, index=False)

print(f"\n[INFO] Saved Section 3.4 Table 1 (HD prevalence by categorical risk factors) to:")
print(table_path)
print(f"[INFO] Suggested APA label: Table X. {TABLE_TITLE}.")

## 4. Data Preparation for Modeling

We:

1. Define **X** (predictors) and **y** (HeartDisease_binary).
2. Split into **train** and **test** sets with stratification.
3. Build a **ColumnTransformer** to:
   - Impute numeric and categorical missingness
   - Standardize numeric features
   - One-hot encode categorical features

This preprocessor will be re-used in all model pipelines to avoid data leakage.

In [ ]:
# --- 4.1 & 4.2. Define predictors/target and train-test split with APA-style exports ---

section_id = "Section4_1"
TABLE_TITLE = "Summary of train–test split and class balance for heart disease outcome"

# --- 4.1. Define predictors and target ---
FEATURE_COLUMNS = numeric_features + categorical_features
TARGET_COLUMN = "HeartDisease_binary"

X = df[FEATURE_COLUMNS].copy()
y = df[TARGET_COLUMN].copy()

# --- 4.2. Train-test split (stratified) ---
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,           # 80/20 split
    stratify=y,
    random_state=SEEDS["train_test"]
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

# Also return shapes like original code
(X_train.shape, X_test.shape)

# -------------------------------------------------------------------------
# APA-style metrics table for train/test split and class balance
#   File: Section4_1_Table1_TrainTest_Split_Summary.csv
# -------------------------------------------------------------------------
def dataset_summary(label, X_subset, y_subset):
    n = X_subset.shape[0]
    p = X_subset.shape[1]
    n_pos = int((y_subset == 1).sum())
    n_neg = int((y_subset == 0).sum())
    pct_pos = (n_pos / n * 100) if n > 0 else float("nan")
    return {
        "Dataset": label,
        "Number of observations (n)": n,
        "Number of predictors (p)": p,
        "Number with heart disease (n = 1)": n_pos,
        "Number without heart disease (n = 0)": n_neg,
        "Heart disease prevalence (%)": round(pct_pos, 2),
        "Target variable name": TARGET_COLUMN,
    }

summary_rows = [
    dataset_summary("Full dataset", X, y),
    dataset_summary("Training set", X_train, y_train),
    dataset_summary("Test set", X_test, y_test),
]

split_summary_df = pd.DataFrame(summary_rows)

table_filename = f"{section_id}_Table1_TrainTest_Split_Summary.csv"
table_path = os.path.join(METRICS_DIR, table_filename)
split_summary_df.to_csv(table_path, index=False)

print(f"\n[INFO] Saved Section 4.1 Table 1 (train–test split summary) to:")
print(table_path)
print(f"[INFO] Suggested APA label: Table X. {TABLE_TITLE}.")

In [ ]:
# --- 4.3. Preprocessing pipelines with APA-style exports ---

section_id   = "Section4_3"
TABLE_TITLE  = "Summary of preprocessing steps for numeric and categorical predictors"

# Numeric: median imputation + standardization
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

# Categorical: most frequent imputation + one-hot encoding
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

# ColumnTransformer to apply the above
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# -------------------------------------------------------------------------
# APA-style metrics table describing preprocessing for each feature
#   File: Section4_3_Table1_Preprocessing_Pipeline_Summary.csv
# -------------------------------------------------------------------------
rows = []

for col in numeric_features:
    rows.append({
        "Feature name": col,
        "Feature type": "Numeric",
        "Imputation strategy": "Median",
        "Transformation(s)": "Standardization (z-score scaling)",
    })

for col in categorical_features:
    rows.append({
        "Feature name": col,
        "Feature type": "Categorical",
        "Imputation strategy": "Most frequent category",
        "Transformation(s)": "One-hot encoding (with unknown categories ignored)",
    })

preprocessing_summary_df = pd.DataFrame(rows)

table_filename = f"{section_id}_Table1_Preprocessing_Pipeline_Summary.csv"
table_path = os.path.join(METRICS_DIR, table_filename)
preprocessing_summary_df.to_csv(table_path, index=False)

print(f"\n[INFO] Saved Section 4.3 Table 1 (preprocessing summary) to:")
print(table_path)
print(f"[INFO] Suggested APA label: Table X. {TABLE_TITLE}.")

display(preprocessing_summary_df)

# Return/show the preprocessor object as in the original code
preprocessor

In [ ]:
# ---
# 5.1. Statsmodels logistic regression for risk factors
#     with APA-style exports
# ---

section_id = "Section5_1"

TABLE1_TITLE = "Model fit statistics for multivariable logistic regression of self-reported heart disease on key risk factors"
TABLE2_TITLE = "Adjusted logistic regression coefficients for key risk factors"

# We'll build a simplified model including key hypothesized predictors:
# BMI (numeric), AgeCategory, Diabetic, Smoking, PhysicalActivity, GenHealth,
# SleepTime (numeric), plus a few core covariates (Sex, Race, Stroke, DiffWalking).

risk_factor_cols = [
    "BMI",
    "SleepTime",
    "AgeCategory",
    "Diabetic",
    "Smoking",
    "PhysicalActivity",
    "GenHealth",
    "Sex",
    "Race",
    "Stroke",
    "DiffWalking"
]

# Drop missing rows in these columns (statsmodels doesn't handle missing automatically)
df_rf = df[risk_factor_cols + [TARGET_COLUMN]].dropna().copy()

# Convert target to float for statsmodels
y_rf = df_rf[TARGET_COLUMN].astype(float)

# One-hot encode categorical predictors manually (drop_first to avoid full collinearity)
X_rf = pd.get_dummies(df_rf[risk_factor_cols], drop_first=True)

# Add intercept term
X_rf = sm.add_constant(X_rf)

# 🔧 IMPORTANT FIX: ensure all predictors are numeric (float), not object
X_rf = X_rf.astype(float)

print("Design matrix shape:", X_rf.shape)
print("Design matrix dtypes:")
print(X_rf.dtypes.head())

# Fit logistic regression
logit_model = sm.Logit(y_rf, X_rf)
logit_result = logit_model.fit(disp=0)  # disp=0 to suppress iterative output

# -------------------------------------------------------------------------
# APA-style Table 1: Model fit statistics
#   File: Section5_1_Table1_LogisticRegression_ModelFit_RiskFactors.csv
# -------------------------------------------------------------------------
model_fit_rows = [
    {"Statistic": "Number of observations (n)",      "Value": logit_result.nobs},
    {"Statistic": "Degrees of freedom (model)",      "Value": logit_result.df_model},
    {"Statistic": "Degrees of freedom (residual)",   "Value": logit_result.df_resid},
    {"Statistic": "Log-likelihood",                  "Value": logit_result.llf},
    {"Statistic": "Akaike information criterion",    "Value": logit_result.aic},
    {"Statistic": "Bayesian information criterion",  "Value": logit_result.bic},
    {"Statistic": "McFadden pseudo R-squared",       "Value": logit_result.prsquared},
]

model_fit_df = pd.DataFrame(model_fit_rows)

table1_filename = f"{section_id}_Table1_LogisticRegression_ModelFit_RiskFactors.csv"
table1_path = os.path.join(METRICS_DIR, table1_filename)
model_fit_df.to_csv(table1_path, index=False)

print(f"\n[INFO] Saved Section 5.1 Table 1 (logistic regression model fit) to:")
print(table1_path)
print(f"[INFO] Suggested APA label: Table X. {TABLE1_TITLE}.")

display(model_fit_df)

# -------------------------------------------------------------------------
# APA-style Table 2: Coefficients, SE, z, p, 95% CI (log-odds)
#   File: Section5_1_Table2_LogisticRegression_Coefficients_RiskFactors.csv
# -------------------------------------------------------------------------
params   = logit_result.params
bse      = logit_result.bse
z_vals   = logit_result.tvalues  # for Logit, these are z-statistics
p_vals   = logit_result.pvalues
conf_int = logit_result.conf_int()
conf_int.columns = ["CI 2.5% (log-odds)", "CI 97.5% (log-odds)"]

coef_df = pd.DataFrame({
    "Predictor": params.index,
    "Coefficient (log-odds)": params.values,
    "Standard error": bse.values,
    "z statistic": z_vals.values,
    "p value": p_vals.values,
    "CI 2.5% (log-odds)": conf_int["CI 2.5% (log-odds)"].values,
    "CI 97.5% (log-odds)": conf_int["CI 97.5% (log-odds)"].values,
})

table2_filename = f"{section_id}_Table2_LogisticRegression_Coefficients_RiskFactors.csv"
table2_path = os.path.join(METRICS_DIR, table2_filename)
coef_df.to_csv(table2_path, index=False)

print(f"\n[INFO] Saved Section 5.1 Table 2 (logistic regression coefficients) to:")
print(table2_path)
print(f"[INFO] Suggested APA label: Table Y. {TABLE2_TITLE}.")

display(coef_df.head())  # show top rows for a quick glance

# -------------------------------------------------------------------------
# Original summary output (kept for detailed console inspection)
# -------------------------------------------------------------------------
print(logit_result.summary())

In [ ]:
# %%
# --- 5.2. Odds ratios, confidence intervals, and p-values for risk factors
#     with APA-style tables and plot exports ---

section_id    = "Section5_2"
TABLE1_TITLE  = "Adjusted odds ratios for all predictors in the multivariable logistic regression model"
TABLE2_TITLE  = "Adjusted odds ratios for statistically significant predictors (p < .05)"
FIGURE_TITLE  = "Adjusted odds ratios for top predictors of self-reported heart disease"

# Extract log-odds, confidence intervals, and p-values from the fitted model
params = logit_result.params
conf   = logit_result.conf_int()
conf.columns = ["lower", "upper"]
p_values = logit_result.pvalues

# Convert to odds ratios
odds_ratios = np.exp(params)
ci_lower    = np.exp(conf["lower"])
ci_upper    = np.exp(conf["upper"])

or_table = pd.DataFrame({
    "OR": odds_ratios,
    "CI_lower": ci_lower,
    "CI_upper": ci_upper,
    "p_value": p_values
})

# Sort by OR magnitude (absolute distance from 1) to highlight strongest effects
or_table["abs_log_OR"] = np.abs(np.log(or_table["OR"]))
or_table_sorted = or_table.sort_values("abs_log_OR", ascending=False)

# Optional: drop the intercept ("const") from exported tables for interpretability
or_table_sorted_no_const = or_table_sorted.drop(index=["const"], errors="ignore")

# -------------------------------------------------------------------------
# Display in notebook (top 25 & top 25 significant), same as original
# -------------------------------------------------------------------------
print("Top 25 predictors by |log(OR)|:")
display(or_table_sorted_no_const.head(25))

# Filter to statistically significant predictors at alpha = 0.05
significant_or = or_table_sorted_no_const[or_table_sorted_no_const["p_value"] < 0.05]

print("\nSignificant predictors (p < 0.05):")
display(significant_or.head(25))

# -------------------------------------------------------------------------
# APA-style Table 1: All predictors (OR, 95% CI, p)
#   File: Section5_2_Table1_LogisticRegression_OddsRatios_AllPredictors.csv
# -------------------------------------------------------------------------
or_table_export = or_table_sorted_no_const.reset_index().rename(
    columns={
        "index": "Predictor",
        "OR": "Odds ratio (OR)",
        "CI_lower": "95% CI lower",
        "CI_upper": "95% CI upper",
        "p_value": "p value",
        "abs_log_OR": "|log(OR)|",
    }
)

# Round for readability in the table
or_table_export["Odds ratio (OR)"] = or_table_export["Odds ratio (OR)"].round(3)
or_table_export["95% CI lower"]   = or_table_export["95% CI lower"].round(3)
or_table_export["95% CI upper"]   = or_table_export["95% CI upper"].round(3)
or_table_export["p value"]        = or_table_export["p value"].round(4)
or_table_export["|log(OR)|"]      = or_table_export["|log(OR)|"].round(3)

table1_filename = f"{section_id}_Table1_LogisticRegression_OddsRatios_AllPredictors.csv"
table1_path = os.path.join(METRICS_DIR, table1_filename)
or_table_export.to_csv(table1_path, index=False)

print(f"\n[INFO] Saved Section 5.2 Table 1 (all predictors' odds ratios) to:")
print(table1_path)
print(f"[INFO] Suggested APA label: Table X. {TABLE1_TITLE}.")

# -------------------------------------------------------------------------
# APA-style Table 2: Significant predictors only (p < .05)
#   File: Section5_2_Table2_LogisticRegression_OddsRatios_SignificantPredictors.csv
# -------------------------------------------------------------------------
sig_or_export = significant_or.reset_index().rename(
    columns={
        "index": "Predictor",
        "OR": "Odds ratio (OR)",
        "CI_lower": "95% CI lower",
        "CI_upper": "95% CI upper",
        "p_value": "p value",
        "abs_log_OR": "|log(OR)|",
    }
)

sig_or_export["Odds ratio (OR)"] = sig_or_export["Odds ratio (OR)"].round(3)
sig_or_export["95% CI lower"]   = sig_or_export["95% CI lower"].round(3)
sig_or_export["95% CI upper"]   = sig_or_export["95% CI upper"].round(3)
sig_or_export["p value"]        = sig_or_export["p value"].round(4)
sig_or_export["|log(OR)|"]      = sig_or_export["|log(OR)|"].round(3)

table2_filename = f"{section_id}_Table2_LogisticRegression_OddsRatios_SignificantPredictors.csv"
table2_path = os.path.join(METRICS_DIR, table2_filename)
sig_or_export.to_csv(table2_path, index=False)

print(f"\n[INFO] Saved Section 5.2 Table 2 (significant predictors' odds ratios) to:")
print(table2_path)
print(f"[INFO] Suggested APA label: Table Y. {TABLE2_TITLE}.")

# -------------------------------------------------------------------------
# Plot: Top predictors by |log(OR)| – bar chart of odds ratios
#   File: Section5_2_Plot1_OddsRatios_TopPredictors.png
# -------------------------------------------------------------------------
top_n = 20
top_or = or_table_export.head(top_n)

plt.figure(figsize=(8, 6))
sns.barplot(
    data=top_or,
    x="Odds ratio (OR)",
    y="Predictor",
    orient="h"
)
plt.axvline(1.0, linestyle="--", color="gray")
plt.xlabel("Odds ratio (logistic regression model)")
plt.ylabel("Predictor")
plt.title(FIGURE_TITLE)
plt.tight_layout()

figure_filename = f"{section_id}_Plot1_OddsRatios_TopPredictors.png"
figure_path = os.path.join(PLOTS_DIR, figure_filename)
plt.savefig(figure_path, dpi=300, bbox_inches="tight")

print(f"[INFO] Saved Section 5.2 Plot 1 (top predictors odds ratios) to:")
print(figure_path)

plt.show()

**Interpretation idea for the paper (not executed here):**

- For H1, we will focus on:
  - The sign and magnitude of ORs (e.g., OR > 1 for diabetes, smoking, older age categories, worse GenHealth).
  - Confidence intervals that do **not** cross 1.
  - Statistical significance (p < 0.05).
- Together with prevalence plots from EDA, these results allow us to assess whether traditional risk factors and lifestyle factors behave as hypothesized.

## 6. Model Definitions and Hyperparameter Tuning (H2 Setup)

We now prepare three models:

1. **Logistic Regression** (baseline, linear)
2. **Random Forest Classifier** (non-linear ensemble)
3. **XGBoost Classifier** (non-linear gradient boosting)

Each model is wrapped in a `Pipeline(preprocessor -> classifier)` and tuned with
`RandomizedSearchCV` using stratified K-fold cross-validation, optimizing **AUROC**.

Class imbalance is addressed via:

- `class_weight='balanced'` for Logistic Regression and Random Forest.
- `scale_pos_weight` for XGBoost (ratio of negative to positive class).

In [ ]:
# --- 6.1. Compute class imbalance ratio (for XGBoost scale_pos_weight) with APA-style export ---

section_id   = "Section6_1"
TABLE_TITLE  = "Class imbalance in the training set and scale_pos_weight used for XGBoost"

# Compute counts in the training set
pos = int(y_train.sum())
neg = int(len(y_train) - pos)
scale_pos_weight = neg / pos

print(f"Positive cases in train: {pos}, negative: {neg}, scale_pos_weight: {scale_pos_weight:.2f}")

# Also compute prevalence for documentation
n_train = len(y_train)
prevalence_prop = pos / n_train
prevalence_pct  = prevalence_prop * 100

# -------------------------------------------------------------------------
# APA-style table for class imbalance and scale_pos_weight
#   File: Section6_1_Table1_ClassImbalance_TrainSet_for_XGBoost.csv
# -------------------------------------------------------------------------
imbalance_df = pd.DataFrame([{
    "Dataset": "Training set",
    "Number of observations (n)": n_train,
    "Number with heart disease (n = 1)": pos,
    "Number without heart disease (n = 0)": neg,
    "Heart disease prevalence (proportion)": round(prevalence_prop, 5),
    "Heart disease prevalence (%)": round(prevalence_pct, 3),
    "Class imbalance ratio (neg:pos)": round(neg / pos, 3),
    "scale_pos_weight (used for XGBoost)": round(scale_pos_weight, 3),
}])

table_filename = f"{section_id}_Table1_ClassImbalance_TrainSet_for_XGBoost.csv"
table_path = os.path.join(METRICS_DIR, table_filename)
imbalance_df.to_csv(table_path, index=False)

print(f"\n[INFO] Saved Section 6.1 Table 1 (class imbalance for XGBoost) to:")
print(table_path)
print(f"[INFO] Suggested APA label: Table X. {TABLE_TITLE}.")


In [ ]:
# --- 6.2. Define model pipelines with APA-style model summary export ---

section_id  = "Section6_2"
TABLE_TITLE = "Initial model specifications and class imbalance handling for logistic regression, random forest, and XGBoost"

# 1) Logistic Regression (baseline)
log_reg_clf = LogisticRegression(
    penalty="l2",
    solver="lbfgs",
    max_iter=1000,
    class_weight="balanced",        # handle imbalance
    random_state=SEEDS["log_reg"]
)

log_reg_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", log_reg_clf)
    ]
)

# 2) Random Forest
rf_clf = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",        # handle imbalance
    n_jobs=-1,
    random_state=SEEDS["rf"]
)

rf_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", rf_clf)
    ]
)

# 3) XGBoost
xgb_clf = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    use_label_encoder=False,
    n_estimators=300,
    learning_rate=0.1,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,  # from Section 6.1
    n_jobs=-1,
    random_state=SEEDS["xgb"]
)

xgb_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", xgb_clf)
    ]
)

# -------------------------------------------------------------------------
# APA-style metrics table summarizing model definitions
#   File: Section6_2_Table1_Model_Pipelines_Summary.csv
# -------------------------------------------------------------------------
model_rows = [
    {
        "Model name": "Logistic regression",
        "Model type": "Penalized logistic regression (L2)",
        "Pipeline components": "Preprocessing (imputation, scaling, one-hot encoding) → LogisticRegression",
        "Key initial hyperparameters": (
            "penalty='l2'; solver='lbfgs'; max_iter=1000"
        ),
        "Class imbalance handling": "class_weight='balanced'"
    },
    {
        "Model name": "Random forest",
        "Model type": "Ensemble of decision trees (RandomForestClassifier)",
        "Pipeline components": "Preprocessing (imputation, scaling, one-hot encoding) → RandomForestClassifier",
        "Key initial hyperparameters": (
            "n_estimators=200; max_depth=None (default); "
            "min_samples_split=2; min_samples_leaf=1"
        ),
        "Class imbalance handling": "class_weight='balanced'"
    },
    {
        "Model name": "XGBoost",
        "Model type": "Gradient-boosted decision trees (XGBClassifier)",
        "Pipeline components": "Preprocessing (imputation, scaling, one-hot encoding) → XGBClassifier",
        "Key initial hyperparameters": (
            "n_estimators=300; learning_rate=0.10; max_depth=5; "
            "subsample=0.8; colsample_bytree=0.8; eval_metric='logloss'"
        ),
        "Class imbalance handling": "scale_pos_weight set to neg:pos ratio in training data"
    },
]

model_summary_df = pd.DataFrame(model_rows)

table_filename = f"{section_id}_Table1_Model_Pipelines_Summary.csv"
table_path = os.path.join(METRICS_DIR, table_filename)
model_summary_df.to_csv(table_path, index=False)

print(f"\n[INFO] Saved Section 6.2 Table 1 (model pipelines summary) to:")
print(table_path)
print(f"[INFO] Suggested APA label: Table X. {TABLE_TITLE}.")

display(model_summary_df)

In [ ]:
# --- 6.3. Hyperparameter search spaces with APA-style export ---

section_id  = "Section6_3"
TABLE_TITLE = "Hyperparameter search spaces for logistic regression, random forest, and XGBoost"

# Original search spaces
log_reg_param_distributions = {
    "model__C": np.logspace(-3, 2, 20),    # regularization strength
}

rf_param_distributions = {
    "model__n_estimators": [200, 400, 600],
    "model__max_depth": [None, 5, 10, 20],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2", 0.3, 0.5],
}

xgb_param_distributions = {
    "model__n_estimators": [200, 400, 600],
    "model__max_depth": [3, 4, 5, 6],
    "model__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "model__subsample": [0.7, 0.8, 0.9],
    "model__colsample_bytree": [0.7, 0.8, 0.9],
    "model__gamma": [0, 0.1, 0.3],
    # scale_pos_weight is fixed from imbalance ratio
}

# -------------------------------------------------------------------------
# APA-style table summarizing the search spaces
#   File: Section6_3_Table1_Hyperparameter_SearchSpaces.csv
# -------------------------------------------------------------------------
rows = []

# Logistic regression
rows.append({
    "Model name": "Logistic regression",
    "Hyperparameter": "C (L2 regularization strength)",
    "Parameter path (pipeline)": "model__C",
    "Search space": "logspace(10^-3, 10^2) with 20 values",
    "Notes": "Controls strength of L2 penalty; larger C = weaker regularization."
})

# Random forest
rows.append({
    "Model name": "Random forest",
    "Hyperparameter": "Number of trees",
    "Parameter path (pipeline)": "model__n_estimators",
    "Search space": "[200, 400, 600]",
    "Notes": "More trees can improve performance but increase computation."
})
rows.append({
    "Model name": "Random forest",
    "Hyperparameter": "Maximum tree depth",
    "Parameter path (pipeline)": "model__max_depth",
    "Search space": "[None, 5, 10, 20]",
    "Notes": "None = full growth; deeper trees capture more complexity but risk overfitting."
})
rows.append({
    "Model name": "Random forest",
    "Hyperparameter": "Minimum samples to split",
    "Parameter path (pipeline)": "model__min_samples_split",
    "Search space": "[2, 5, 10]",
    "Notes": "Larger values make trees more conservative."
})
rows.append({
    "Model name": "Random forest",
    "Hyperparameter": "Minimum samples per leaf",
    "Parameter path (pipeline)": "model__min_samples_leaf",
    "Search space": "[1, 2, 4]",
    "Notes": "Controls the minimum number of samples in terminal leaves."
})
rows.append({
    "Model name": "Random forest",
    "Hyperparameter": "Max features per split",
    "Parameter path (pipeline)": "model__max_features",
    "Search space": "['sqrt', 'log2', 0.3, 0.5]",
    "Notes": "Fraction or function of features considered at each split."
})

# XGBoost
rows.append({
    "Model name": "XGBoost",
    "Hyperparameter": "Number of boosting rounds",
    "Parameter path (pipeline)": "model__n_estimators",
    "Search space": "[200, 400, 600]",
    "Notes": "More estimators can increase performance but risk overfitting."
})
rows.append({
    "Model name": "XGBoost",
    "Hyperparameter": "Maximum tree depth",
    "Parameter path (pipeline)": "model__max_depth",
    "Search space": "[3, 4, 5, 6]",
    "Notes": "Deeper trees capture more complex patterns but may overfit."
})
rows.append({
    "Model name": "XGBoost",
    "Hyperparameter": "Learning rate (eta)",
    "Parameter path (pipeline)": "model__learning_rate",
    "Search space": "[0.01, 0.05, 0.10, 0.20]",
    "Notes": "Smaller values make learning slower but more stable."
})
rows.append({
    "Model name": "XGBoost",
    "Hyperparameter": "Subsample (rows)",
    "Parameter path (pipeline)": "model__subsample",
    "Search space": "[0.7, 0.8, 0.9]",
    "Notes": "Row subsampling per tree; helps regularization."
})
rows.append({
    "Model name": "XGBoost",
    "Hyperparameter": "Column subsample (colsample_bytree)",
    "Parameter path (pipeline)": "model__colsample_bytree",
    "Search space": "[0.7, 0.8, 0.9]",
    "Notes": "Column subsampling per tree; helps regularization."
})
rows.append({
    "Model name": "XGBoost",
    "Hyperparameter": "Gamma (min loss reduction)",
    "Parameter path (pipeline)": "model__gamma",
    "Search space": "[0, 0.1, 0.3]",
    "Notes": "Minimum loss reduction required to make a further partition on a leaf node."
})

hyperparam_table_df = pd.DataFrame(rows)

table_filename = f"{section_id}_Table1_Hyperparameter_SearchSpaces.csv"
table_path = os.path.join(METRICS_DIR, table_filename)
hyperparam_table_df.to_csv(table_path, index=False)

print(f"\n[INFO] Saved Section 6.3 Table 1 (hyperparameter search spaces) to:")
print(table_path)
print(f"[INFO] Suggested APA label: Table X. {TABLE_TITLE}.")

display(hyperparam_table_df)

In [ ]:
# --- 6.4. Helper: RandomizedSearchCV wrapper with APA-style exports ---

def tune_model_with_random_search(
    pipeline,
    param_distributions,
    model_name: str,
    n_iter: int = 20,
    cv_splits: int = 5
):
    """
    Generic helper to tune a model with RandomizedSearchCV on AUROC.
    Returns fitted RandomizedSearchCV object and saves an APA-style
    table summarizing the best hyperparameters and AUROC.
    """
    print(f"\n=== Hyperparameter tuning: {model_name} ===")
    cv = StratifiedKFold(
        n_splits=cv_splits,
        shuffle=True,
        random_state=SEEDS["cv"]
    )
    
    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_distributions,
        n_iter=n_iter,
        scoring="roc_auc",
        n_jobs=-1,
        cv=cv,
        verbose=2,
        random_state=SEEDS["cv"],
        refit=True,
    )
    
    search.fit(X_train, y_train)
    print(f"Best AUROC ({model_name}): {search.best_score_:.4f}")
    print("Best params:")
    print(search.best_params_)
    
    # ---------------------------------------------------------------------
    # APA-style metrics export: best params + best AUROC
    #   Files:
    #     Section6_4_Table1_LogisticRegression_RandomSearch_BestParams.csv
    #     Section6_4_Table2_RandomForest_RandomSearch_BestParams.csv
    #     Section6_4_Table3_XGBoost_RandomSearch_BestParams.csv
    # ---------------------------------------------------------------------
    section_id  = "Section6_4"
    TABLE_TITLE = (
        "Best hyperparameters and mean cross-validated AUROC from "
        "RandomizedSearchCV for each model"
    )
    
    # Map model_name → table index for consistent naming
    table_index_map = {
        "Logistic regression": 1,
        "Random forest": 2,
        "XGBoost": 3,
    }
    table_idx = table_index_map.get(model_name, 99)  # fallback index if name differs
    
    # Build a single-row DataFrame: metadata + best hyperparameters
    best_params = search.best_params_
    row = {
        "Model name": model_name,
        "Number of CV folds": cv_splits,
        "Number of random search iterations": n_iter,
        "Best mean cross-validated AUROC": round(search.best_score_, 4),
    }
    
    # Add each best hyperparameter as its own column
    for param_path, value in best_params.items():
        row[f"Best value: {param_path}"] = value
    
    best_params_df = pd.DataFrame([row])
    
    table_filename = f"{section_id}_Table{table_idx}_{model_name.replace(' ', '')}_RandomSearch_BestParams.csv"
    table_path = os.path.join(METRICS_DIR, table_filename)
    best_params_df.to_csv(table_path, index=False)
    
    print(f"[INFO] Saved Section 6.4 Table {table_idx} (best params for {model_name}) to:")
    print(table_path)
    print(f"[INFO] Suggested APA label (for combined table in text): Table X. {TABLE_TITLE}.")
    
    display(best_params_df)
    
    return search

In [ ]:
# --- 6.5. Run hyperparameter tuning for all models and summarize AUROC ---

# Run tuning for each model
log_reg_search = tune_model_with_random_search(
    log_reg_pipeline,
    log_reg_param_distributions,
    model_name="Logistic regression"   # match names used in 6.4 mapping
)

rf_search = tune_model_with_random_search(
    rf_pipeline,
    rf_param_distributions,
    model_name="Random forest"
)

xgb_search = tune_model_with_random_search(
    xgb_pipeline,
    xgb_param_distributions,
    model_name="XGBoost"
)

# Extract best pipelines for subsequent evaluation
best_log_reg = log_reg_search.best_estimator_
best_rf      = rf_search.best_estimator_
best_xgb     = xgb_search.best_estimator_

print("\n[INFO] Extracted best pipelines for all models.")
print("  best_log_reg:", best_log_reg)
print("  best_rf:", best_rf)
print("  best_xgb:", best_xgb)

# -------------------------------------------------------------------------
# APA-style comparison table of best CV AUROC across models
#   File: Section6_5_Table1_TunedModels_CV_AUROC_Comparison.csv
# -------------------------------------------------------------------------
section_id  = "Section6_5"
TABLE_TITLE = "Comparison of tuned models based on cross-validated AUROC"

cv_rows = [
    {
        "Model name": "Logistic regression",
        "Best mean cross-validated AUROC": round(log_reg_search.best_score_, 4),
    },
    {
        "Model name": "Random forest",
        "Best mean cross-validated AUROC": round(rf_search.best_score_, 4),
    },
    {
        "Model name": "XGBoost",
        "Best mean cross-validated AUROC": round(xgb_search.best_score_, 4),
    },
]

cv_comparison_df = pd.DataFrame(cv_rows)

table_filename = f"{section_id}_Table1_TunedModels_CV_AUROC_Comparison.csv"
table_path = os.path.join(METRICS_DIR, table_filename)
cv_comparison_df.to_csv(table_path, index=False)

print(f"\n[INFO] Saved Section 6.5 Table 1 (tuned models CV AUROC comparison) to:")
print(table_path)
print(f"[INFO] Suggested APA label: Table X. {TABLE_TITLE}.")

display(cv_comparison_df)

## 7. Cross-Validation Comparison (H2 – Model Performance)

To formally test **H2**, we compare cross-validated **AUROC** for:

- Logistic Regression (baseline)
- Random Forest (non-linear)
- XGBoost (non-linear)

using **the same CV splits**, and then conduct **paired t-tests** on per-fold AUROCs.

In [ ]:
# ---
# 7.1. Cross-validated AUROC for best models on train set
#     with APA-style tables and plot exports
# ---

section_id   = "Section7_1"
TABLE1_TITLE = "Per-fold cross-validated AUROC for tuned models on the training set"
TABLE2_TITLE = "Summary of cross-validated AUROC for tuned models on the training set"
FIGURE_TITLE = "Cross-validated AUROC distributions by model (training set)"

def cross_val_auroc(model, model_name: str, cv_splits: int = 5):
    """
    Compute cross-validated AUROC for a given fitted model using new CV splits.
    Returns array of AUROC scores (one per fold).
    """
    print(f"\n=== Cross-validated AUROC for {model_name} ===")
    cv = StratifiedKFold(
        n_splits=cv_splits,
        shuffle=True,
        random_state=SEEDS["cv"]
    )
    cv_results = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1,
        return_estimator=False
    )
    aucs = cv_results["test_score"]
    print(f"Mean AUROC: {aucs.mean():.4f} ± {aucs.std():.4f}")
    return aucs

# Run CV AUROC for each tuned model (using consistent model names)
aucs_log_reg = cross_val_auroc(best_log_reg, "Logistic regression")
aucs_rf      = cross_val_auroc(best_rf,      "Random forest")
aucs_xgb     = cross_val_auroc(best_xgb,     "XGBoost")

# -------------------------------------------------------------------------
# APA-style Table 1: per-fold AUROC values for each model
#   File: Section7_1_Table1_CV_AUROC_PerFold_byModel.csv
# -------------------------------------------------------------------------
n_folds = len(aucs_log_reg)  # assumes same CV splits across models

fold_indices = np.arange(1, n_folds + 1)

perfold_rows = []

for fold, score in zip(fold_indices, aucs_log_reg):
    perfold_rows.append({
        "Model name": "Logistic regression",
        "Fold": fold,
        "AUROC": score,
    })

for fold, score in zip(fold_indices, aucs_rf):
    perfold_rows.append({
        "Model name": "Random forest",
        "Fold": fold,
        "AUROC": score,
    })

for fold, score in zip(fold_indices, aucs_xgb):
    perfold_rows.append({
        "Model name": "XGBoost",
        "Fold": fold,
        "AUROC": score,
    })

cv_perfold_df = pd.DataFrame(perfold_rows)

table1_filename = f"{section_id}_Table1_CV_AUROC_PerFold_byModel.csv"
table1_path = os.path.join(METRICS_DIR, table1_filename)
cv_perfold_df.to_csv(table1_path, index=False)

print(f"\n[INFO] Saved Section 7.1 Table 1 (per-fold AUROC by model) to:")
print(table1_path)
print(f"[INFO] Suggested APA label: Table X. {TABLE1_TITLE}.")

display(cv_perfold_df)

# -------------------------------------------------------------------------
# APA-style Table 2: summary AUROC stats by model (mean, SD, min, max)
#   File: Section7_1_Table2_CV_AUROC_Summary_byModel.csv
# -------------------------------------------------------------------------
summary_rows = []
for model_name, aucs in [
    ("Logistic regression", aucs_log_reg),
    ("Random forest",       aucs_rf),
    ("XGBoost",             aucs_xgb),
]:
    summary_rows.append({
        "Model name": model_name,
        "Mean AUROC": np.mean(aucs),
        "Std. deviation AUROC": np.std(aucs, ddof=1),
        "Minimum AUROC": np.min(aucs),
        "Maximum AUROC": np.max(aucs),
    })

cv_summary_df = pd.DataFrame(summary_rows)

# Round for nicer table output
cv_summary_df["Mean AUROC"] = cv_summary_df["Mean AUROC"].round(4)
cv_summary_df["Std. deviation AUROC"] = cv_summary_df["Std. deviation AUROC"].round(4)
cv_summary_df["Minimum AUROC"] = cv_summary_df["Minimum AUROC"].round(4)
cv_summary_df["Maximum AUROC"] = cv_summary_df["Maximum AUROC"].round(4)

table2_filename = f"{section_id}_Table2_CV_AUROC_Summary_byModel.csv"
table2_path = os.path.join(METRICS_DIR, table2_filename)
cv_summary_df.to_csv(table2_path, index=False)

print(f"\n[INFO] Saved Section 7.1 Table 2 (summary AUROC by model) to:")
print(table2_path)
print(f"[INFO] Suggested APA label: Table Y. {TABLE2_TITLE}.")

display(cv_summary_df)

# -------------------------------------------------------------------------
# Plot: AUROC distribution by model (boxplot)
#   File: Section7_1_Plot1_CV_AUROC_Distribution_byModel.png
# -------------------------------------------------------------------------
plt.figure(figsize=(7, 5))
sns.boxplot(
    data=cv_perfold_df,
    x="Model name",
    y="AUROC"
)
plt.xlabel("Model")
plt.ylabel("Area under the ROC curve (AUROC)")
plt.title(FIGURE_TITLE)
plt.tight_layout()

figure_filename = f"{section_id}_Plot1_CV_AUROC_Distribution_byModel.png"
figure_path = os.path.join(PLOTS_DIR, figure_filename)
plt.savefig(figure_path, dpi=300, bbox_inches="tight")

print(f"\n[INFO] Saved Section 7.1 Plot 1 (CV AUROC distribution by model) to:")
print(figure_path)

plt.show()

In [ ]:
# --- 7.2. Paired t-tests of AUROC between models (per-fold) with APA-style export ---

section_id   = "Section7_2"
TABLE_TITLE  = "Paired t-test comparisons of cross-validated AUROC between tuned models"

ttest_results = []

def compare_models_ttest(aucs_a, aucs_b, name_a: str, name_b: str):
    """
    Perform a paired t-test on AUROC arrays from two models.
    Also appends results to ttest_results for APA-style export.
    """
    t_stat, p_val = stats.ttest_rel(aucs_a, aucs_b)
    diff_mean = np.mean(aucs_b - aucs_a)
    
    print(f"\n=== AUROC comparison: {name_b} vs {name_a} ===")
    print(f"Mean AUROC {name_a}: {aucs_a.mean():.4f}")
    print(f"Mean AUROC {name_b}: {aucs_b.mean():.4f}")
    print(f"Mean difference ( {name_b} - {name_a} ): {diff_mean:.4f}")
    print(f"Paired t-test: t = {t_stat:.4f}, p = {p_val:.4g}")
    
    # Store results for export
    ttest_results.append({
        "Model A": name_a,
        "Model B": name_b,
        "Mean AUROC (Model A)": aucs_a.mean(),
        "Mean AUROC (Model B)": aucs_b.mean(),
        "Mean difference (B - A)": diff_mean,
        "t statistic": t_stat,
        "p value": p_val,
        "Number of folds (n)": len(aucs_a),
    })

# Use consistent naming with earlier sections ("Logistic regression", "Random forest", "XGBoost")
compare_models_ttest(aucs_log_reg, aucs_rf,  "Logistic regression", "Random forest")
compare_models_ttest(aucs_log_reg, aucs_xgb, "Logistic regression", "XGBoost")
compare_models_ttest(aucs_rf,      aucs_xgb, "Random forest",       "XGBoost")

# -------------------------------------------------------------------------
# APA-style table export for all pairwise t-tests
#   File: Section7_2_Table1_PairedTTests_AUROC_Comparisons.csv
# -------------------------------------------------------------------------
ttest_df = pd.DataFrame(ttest_results)

# Round for readability
ttest_df["Mean AUROC (Model A)"]      = ttest_df["Mean AUROC (Model A)"].round(4)
ttest_df["Mean AUROC (Model B)"]      = ttest_df["Mean AUROC (Model B)"].round(4)
ttest_df["Mean difference (B - A)"]   = ttest_df["Mean difference (B - A)"].round(4)
ttest_df["t statistic"]               = ttest_df["t statistic"].round(4)
ttest_df["p value"]                   = ttest_df["p value"].round(4)

table_filename = f"{section_id}_Table1_PairedTTests_AUROC_Comparisons.csv"
table_path = os.path.join(METRICS_DIR, table_filename)
ttest_df.to_csv(table_path, index=False)

print(f"\n[INFO] Saved Section 7.2 Table 1 (paired t-tests of AUROC) to:")
print(table_path)
print(f"[INFO] Suggested APA label: Table X. {TABLE_TITLE}.")

display(ttest_df)

For **H2**, we will interpret:

- Whether Random Forest / XGBoost show **higher mean AUROC** than Logistic Regression.
- Whether paired t-tests indicate statistically significant improvements (e.g., p < 0.05).

These results directly address the hypothesis that non-linear models outperform logistic regression.

## 8. Final Test-Set Evaluation and Curves

We now evaluate all three tuned models on the **held-out test set**:

- AUROC
- Average Precision (AUPRC)
- Accuracy, Precision, Recall, F1
- Confusion matrix
- ROC and Precision–Recall curves

This section supports both H1 (via performance using risk factors) and H2 (via
comparative performance of model families).

In [ ]:
# ---
# 8.1. Helper: evaluate a model on the test set
#      with APA-style tables and plot exports
# ---

section_id = "Section8_1"

def evaluate_on_test(model, model_name: str):
    """
    Compute metrics and plot ROC/PR curves for a model on the test set.
    Also saves confusion matrix and plots with APA-style filenames.
    """
    print(f"\n{'='*80}")
    print(f"Test-set evaluation: {model_name}")
    print('='*80)
    
    # Predicted probabilities (positive class)
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)  # threshold can be tuned
    
    # Metrics
    auc  = roc_auc_score(y_test, y_proba)
    ap   = average_precision_score(y_test, y_proba)
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)
    
    print(f"AUROC: {auc:.4f}")
    print(f"Average Precision (AUPRC): {ap:.4f}")
    print(f"Accuracy: {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall (Sensitivity): {rec:.4f}")
    print(f"F1-score: {f1:.4f}")
    
    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    cm_df = pd.DataFrame(
        cm,
        index=["True 0 (No heart disease)", "True 1 (Heart disease)"],
        columns=["Predicted 0", "Predicted 1"]
    )
    print("\nConfusion Matrix:")
    display(cm_df)
    
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, digits=3))
    
    # ---------------------------------------------------------------------
    # Save confusion matrix as APA-style table (per model)
    # ---------------------------------------------------------------------
    table_index_map = {
        "Logistic regression": 2,
        "Random forest": 3,
        "XGBoost": 4,
    }
    table_idx = table_index_map.get(model_name, 99)
    
    safe_model_tag = model_name.replace(" ", "")
    cm_filename = f"{section_id}_Table{table_idx}_ConfusionMatrix_{safe_model_tag}.csv"
    cm_path = os.path.join(METRICS_DIR, cm_filename)
    cm_df.to_csv(cm_path, index=True)
    
    print(f"[INFO] Saved confusion matrix table for {model_name} to:")
    print(cm_path)
    
    # ---------------------------------------------------------------------
    # ROC curve
    # ---------------------------------------------------------------------
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    plt.figure(figsize=(6, 4))
    plt.plot(fpr, tpr, label=f"{model_name} (AUROC = {auc:.3f})")
    plt.plot([0, 1], [0, 1], "k--", label="Chance")
    plt.xlabel("False positive rate")
    plt.ylabel("True positive rate")
    plt.title(f"ROC curve for {model_name} on the test set")
    plt.legend()
    plt.tight_layout()
    
    # Map model → base plot indices (ROC = base, PR = base+1)
    plot_index_map = {
        "Logistic regression": 1,
        "Random forest": 3,
        "XGBoost": 5,
    }
    base_idx = plot_index_map.get(model_name, 99)
    
    roc_filename = f"{section_id}_Plot{base_idx}_ROC_{safe_model_tag}.png"
    roc_path = os.path.join(PLOTS_DIR, roc_filename)
    plt.savefig(roc_path, dpi=300, bbox_inches="tight")
    
    print(f"[INFO] Saved ROC curve plot for {model_name} to:")
    print(roc_path)
    
    plt.show()
    
    # ---------------------------------------------------------------------
    # Precision–Recall curve
    # ---------------------------------------------------------------------
    prec_curve, rec_curve, _ = precision_recall_curve(y_test, y_proba)
    plt.figure(figsize=(6, 4))
    plt.plot(rec_curve, prec_curve, label=f"{model_name} (AP = {ap:.3f})")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"Precision–Recall curve for {model_name} on the test set")
    plt.legend()
    plt.tight_layout()
    
    pr_filename = f"{section_id}_Plot{base_idx + 1}_PR_{safe_model_tag}.png"
    pr_path = os.path.join(PLOTS_DIR, pr_filename)
    plt.savefig(pr_path, dpi=300, bbox_inches="tight")
    
    print(f"[INFO] Saved Precision–Recall curve plot for {model_name} to:")
    print(pr_path)
    
    plt.show()
    
    return {
        "Model name": model_name,
        "AUROC": auc,
        "AUPRC": ap,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1
    }

# Evaluate each tuned model on the test set
results_log_reg = evaluate_on_test(best_log_reg, "Logistic regression")
results_rf      = evaluate_on_test(best_rf,      "Random forest")
results_xgb     = evaluate_on_test(best_xgb,     "XGBoost")

# -------------------------------------------------------------------------
# APA-style summary table of test-set metrics across models
#   File: Section8_1_Table1_TestSet_Metrics_byModel.csv
# -------------------------------------------------------------------------
test_results_df = pd.DataFrame([results_log_reg, results_rf, results_xgb])

# Round for clean table output
for col in ["AUROC", "AUPRC", "Accuracy", "Precision", "Recall", "F1"]:
    test_results_df[col] = test_results_df[col].round(4)

print("\nSummary of test-set metrics:")
display(test_results_df)

table1_filename = f"{section_id}_Table1_TestSet_Metrics_byModel.csv"
table1_path = os.path.join(METRICS_DIR, table1_filename)
test_results_df.to_csv(table1_path, index=False)

print(f"\n[INFO] Saved Section 8.1 Table 1 (test-set metrics by model) to:")
print(table1_path)
print("[INFO] Suggested APA label: Table X. Test-set performance of tuned models (AUROC, AUPRC, accuracy, precision, recall, and F1).")

## 9. Model Explainability (Feature Importance & SHAP)

To further address **H1 (risk factors)** and interpret our models, we:

1. Extract feature importance from the best Random Forest and XGBoost models.
2. Use **SHAP** values on XGBoost (tree-based model) to:
   - Identify globally important predictors.
   - Visualize effects of specific features.

Note: SHAP can be computationally expensive on large datasets; we may subsample.

In [ ]:
# ---
# 9.1. Helper: get feature names from ColumnTransformer + OneHotEncoder
#      with APA-style feature listing export
# ---

section_id  = "Section9_1"
TABLE_TITLE = "Final model feature names and their original variables"

def get_feature_names_from_preprocessor(preprocessor, numeric_features, categorical_features):
    """
    Extract final feature names from a ColumnTransformer with numeric and categorical pipelines.
    Returns a flat list of feature names in the order they appear in the model.
    """
    # Numeric features pass through scaler; names are the same
    num_features_out = numeric_features
    
    # Categorical features: need to extract from OneHotEncoder
    cat_pipeline = preprocessor.named_transformers_["cat"]
    ohe = cat_pipeline.named_steps["onehot"]
    cat_features_out = list(ohe.get_feature_names_out(categorical_features))
    
    return num_features_out + cat_features_out

# Use the fitted preprocessor from the best XGBoost pipeline
xgb_preprocessor = best_xgb.named_steps["preprocess"]

feature_names = get_feature_names_from_preprocessor(
    xgb_preprocessor,
    numeric_features,
    categorical_features
)

print("Number of final model features:", len(feature_names))
print("First 10 feature names:")
print(feature_names[:10])

# -------------------------------------------------------------------------
# Build APA-style table: model feature → original variable & type
# -------------------------------------------------------------------------
cat_pipeline = xgb_preprocessor.named_transformers_["cat"]
ohe = cat_pipeline.named_steps["onehot"]

# Reconstruct original-variable mapping for categorical features
orig_var_list = []
feature_type_list = []

# 1) Numeric features
for col in numeric_features:
    orig_var_list.append(col)
    feature_type_list.append("Numeric (standardized)")

# 2) Categorical features – expand each category as a separate feature
#    in the same order as OneHotEncoder would produce them
for var_name, categories in zip(categorical_features, ohe.categories_):
    for _ in categories:
        orig_var_list.append(var_name)
        feature_type_list.append("Categorical (one-hot encoded)")

# Sanity check: lengths should match
assert len(feature_names) == len(orig_var_list), (
    f"Length mismatch: {len(feature_names)} feature names vs "
    f"{len(orig_var_list)} origin entries."
)

feature_metadata_df = pd.DataFrame({
    "Model feature name": feature_names,
    "Original variable": orig_var_list,
    "Variable type / encoding": feature_type_list,
})

# -------------------------------------------------------------------------
# Save APA-style table for Section 9.1
#   File: Section9_1_Table1_Model_Feature_Names_and_Origins.csv
# -------------------------------------------------------------------------
table_filename = f"{section_id}_Table1_Model_Feature_Names_and_Origins.csv"
table_path = os.path.join(METRICS_DIR, table_filename)
feature_metadata_df.to_csv(table_path, index=False)

print(f"\n[INFO] Saved Section 9.1 Table 1 (model feature names and origins) to:")
print(table_path)
print(f"[INFO] Suggested APA label: Table X. {TABLE_TITLE}.")

display(feature_metadata_df.head(15))

In [ ]:
# --- 9.2. Feature importance for Random Forest and XGBoost with APA-style exports ---

section_id = "Section9_2"

def plot_top_features_from_model(
    model,
    feature_names,
    model_name: str,
    top_n: int = 20,
    table_index: int = None,
    plot_index: int = None
):
    """
    Plot top_n features by importance from a tree-based model inside a pipeline,
    and save APA-style table + figure.
    """
    tree_model = model.named_steps["model"]
    importances = tree_model.feature_importances_
    
    importance_df = (
        pd.DataFrame({
            "Model feature name": feature_names,
            "Feature importance (Gini-based)": importances
        })
        .sort_values("Feature importance (Gini-based)", ascending=False)
        .head(top_n)
    )
    
    # ------------------------------------------------------------------
    # Save APA-style table of top features (if table_index provided)
    # ------------------------------------------------------------------
    if table_index is not None:
        safe_model_tag = model_name.replace(" ", "")
        table_filename = (
            f"{section_id}_Table{table_index}_Top{top_n}Features_{safe_model_tag}.csv"
        )
        table_path = os.path.join(METRICS_DIR, table_filename)
        importance_df.to_csv(table_path, index=False)
        
        print(f"\n[INFO] Saved {section_id} Table {table_index} (top {top_n} features for {model_name}) to:")
        print(table_path)
        print(
            f"[INFO] Suggested APA label: Table X. Top {top_n} features ranked by "
            f"Gini-based importance for {model_name}."
        )
    
    # ------------------------------------------------------------------
    # Plot bar chart and save figure (if plot_index provided)
    # ------------------------------------------------------------------
    plt.figure(figsize=(8, 6))
    sns.barplot(
        data=importance_df,
        x="Feature importance (Gini-based)",
        y="Model feature name"
    )
    plt.title(f"Top {top_n} features by importance – {model_name}")
    plt.xlabel("Feature importance (Gini-based)")
    plt.ylabel("")
    plt.tight_layout()
    
    if plot_index is not None:
        safe_model_tag = model_name.replace(" ", "")
        figure_filename = (
            f"{section_id}_Plot{plot_index}_FeatureImportance_{safe_model_tag}.png"
        )
        figure_path = os.path.join(PLOTS_DIR, figure_filename)
        plt.savefig(figure_path, dpi=300, bbox_inches="tight")
        
        print(f"[INFO] Saved {section_id} Plot {plot_index} (feature importance for {model_name}) to:")
        print(figure_path)
    
    plt.show()
    
    return importance_df

# Top 20 features for Random Forest
rf_importance_df = plot_top_features_from_model(
    best_rf,
    feature_names,
    model_name="Random forest",
    top_n=20,
    table_index=1,
    plot_index=1
)

# Top 20 features for XGBoost
xgb_importance_df = plot_top_features_from_model(
    best_xgb,
    feature_names,
    model_name="XGBoost",
    top_n=20,
    table_index=2,
    plot_index=2
)

print("\nRandom forest – top features:")
display(rf_importance_df)

print("\nXGBoost – top features:")
display(xgb_importance_df)

In [ ]:
# --- 9.3. SHAP analysis for XGBoost (global feature importance) with APA-style exports ---

section_id  = "Section9_3"
TABLE_TITLE = "Global SHAP-based feature importance for the tuned XGBoost model"
FIGURE_TITLE = "Global SHAP feature importance (top 20 predictors) for XGBoost"

# Initialize SHAP JS (for notebook interactivity, if supported)
shap.initjs()

# Transform X_train using the preprocessor from the best_xgb pipeline
X_train_encoded = best_xgb.named_steps["preprocess"].transform(X_train)

# For some sklearn/OneHotEncoder setups, this may be sparse. Convert later if needed.

# Underlying XGBoost model
xgb_model_inner = best_xgb.named_steps["model"]

# For speed, sample a subset of rows (e.g., 10,000); adjust as needed
max_shap_samples = 10000
if X_train_encoded.shape[0] > max_shap_samples:
    sample_idx = np.random.choice(
        X_train_encoded.shape[0],
        size=max_shap_samples,
        replace=False
    )
    X_shap = X_train_encoded[sample_idx]
else:
    X_shap = X_train_encoded

# Convert to dense array if sparse
if hasattr(X_shap, "toarray"):
    X_shap = X_shap.toarray()

# Compute SHAP values
explainer = shap.TreeExplainer(xgb_model_inner)
shap_values = explainer(X_shap)

# -------------------------------------------------------------------------
# Global SHAP importance: mean |SHAP| per feature
# -------------------------------------------------------------------------
# shap_values.values is (n_samples, n_features)
shap_abs_mean = np.mean(np.abs(shap_values.values), axis=0)

shap_importance_df = (
    pd.DataFrame({
        "Model feature name": feature_names,
        "Mean |SHAP value| (global importance)": shap_abs_mean
    })
    .sort_values("Mean |SHAP value| (global importance)", ascending=False)
)

# Top 20 for reporting
top_n = 20
shap_top20_df = shap_importance_df.head(top_n)

print(f"\nTop {top_n} features by mean |SHAP| for XGBoost:")
display(shap_top20_df)

# -------------------------------------------------------------------------
# Save APA-style table
#   File: Section9_3_Table1_SHAP_GlobalImportance_XGBoost.csv
# -------------------------------------------------------------------------
table_filename = f"{section_id}_Table1_SHAP_GlobalImportance_XGBoost.csv"
table_path = os.path.join(METRICS_DIR, table_filename)
shap_top20_df.to_csv(table_path, index=False)

print(f"[INFO] Saved Section 9.3 Table 1 (SHAP global importance, top 20) to:")
print(table_path)
print(f"[INFO] Suggested APA label: Table X. {TABLE_TITLE}.")

# -------------------------------------------------------------------------
# Global summary bar plot using SHAP, then save the figure
# -------------------------------------------------------------------------
# Use bar-type SHAP summary plot (global importance)
shap.summary_plot(
    shap_values.values,
    X_shap,
    feature_names=feature_names,
    plot_type="bar",
    max_display=top_n,
    show=False  # let us grab and save the figure before displaying
)

plt.title(FIGURE_TITLE)
plt.tight_layout()

figure_filename = f"{section_id}_Plot1_SHAP_GlobalImportance_XGBoost.png"
figure_path = os.path.join(PLOTS_DIR, figure_filename)
plt.savefig(figure_path, dpi=300, bbox_inches="tight")

print(f"[INFO] Saved Section 9.3 Plot 1 (SHAP global importance bar plot) to:")
print(figure_path)

plt.show()

In [ ]:
# --- 9.4. SHAP summary (beeswarm) for detailed effects with APA-style export ---

section_id   = "Section9_4"
FIGURE_TITLE = "SHAP beeswarm plot for the tuned XGBoost model (top 20 predictors)"

# SHAP beeswarm summary: shows distribution and direction of effects
# Uses shap_values, X_shap, and feature_names from Section 9.3
shap.summary_plot(
    shap_values.values,
    X_shap,
    feature_names=feature_names,
    max_display=20,
    show=False  # allow us to capture and save the figure
)

# Adjust title/layout for APA-style figure
plt.title(FIGURE_TITLE)
plt.tight_layout()

# Save figure
figure_filename = f"{section_id}_Plot1_SHAP_Beeswarm_XGBoost.png"
figure_path = os.path.join(PLOTS_DIR, figure_filename)

fig = plt.gcf()
fig.savefig(figure_path, dpi=300, bbox_inches="tight")

print(f"[INFO] Saved Section 9.4 Plot 1 (SHAP beeswarm for XGBoost) to:")
print(figure_path)
print("[INFO] Suggested APA label: Figure X. SHAP beeswarm plot showing the distribution and direction of feature effects for the tuned XGBoost model.")

plt.show()

For **H1**, we can now:

- Check if features such as **AgeCategory**, **BMI**, **Diabetic**, **Smoking**, **PhysicalActivity**, **GenHealth**, and **SleepTime**:
  - Have large positive coefficients / odds ratios in logistic regression.
  - Appear among the top features in Random Forest / XGBoost importance.
  - Show strong SHAP contributions to predicted risk.

## 10. Subgroup Analyses

To explore fairness and heterogeneity of performance, we:

- Evaluate the best-performing model (likely XGBoost or Random Forest) across
  subgroups defined by:
  - **Sex**
  - **AgeCategory** (original categories)
  - **Race**
  - **Diabetic** status

We compute AUROC and key metrics within each subgroup on the **test set**.

In [ ]:
# ---
# 10.1. Choose a "final" model for subgroup analysis with APA-style summary
# ---

section_id  = "Section10_1"
TABLE_TITLE = "Final model selected for subgroup analyses and its overall test-set performance"

# You may choose whichever has highest AUROC on the test set; here we assume XGBoost.
final_model = best_xgb
final_model_name = "XGBoost (final)"

print(f"Final model selected for subgroup analysis: {final_model_name}")

# For subgroup metrics, we need test-set probabilities once
y_test_proba_final = final_model.predict_proba(X_test)[:, 1]
y_test_pred_final  = (y_test_proba_final >= 0.5).astype(int)

# -------------------------------------------------------------------------
# Compute overall test-set metrics for the final model
# -------------------------------------------------------------------------
auc_final  = roc_auc_score(y_test, y_test_proba_final)
ap_final   = average_precision_score(y_test, y_test_proba_final)
acc_final  = accuracy_score(y_test, y_test_pred_final)
prec_final = precision_score(y_test, y_test_pred_final)
rec_final  = recall_score(y_test, y_test_pred_final)
f1_final   = f1_score(y_test, y_test_pred_final)

print("\nOverall test-set performance for final model:")
print(f"AUROC:   {auc_final:.4f}")
print(f"AUPRC:   {ap_final:.4f}")
print(f"Accuracy:{acc_final:.4f}")
print(f"Precision:{prec_final:.4f}")
print(f"Recall:  {rec_final:.4f}")
print(f"F1-score:{f1_final:.4f}")

# -------------------------------------------------------------------------
# APA-style table: final model choice + global test-set metrics
#   File: Section10_1_Table1_FinalModel_Choice_and_GlobalMetrics.csv
# -------------------------------------------------------------------------
final_model_df = pd.DataFrame([{
    "Final model name": final_model_name,
    "Model family": "XGBoost (gradient-boosted decision trees)",
    "AUROC (test set)": round(auc_final, 4),
    "AUPRC (test set)": round(ap_final, 4),
    "Accuracy (test set)": round(acc_final, 4),
    "Precision (test set)": round(prec_final, 4),
    "Recall (test set)": round(rec_final, 4),
    "F1-score (test set)": round(f1_final, 4),
}])

table_filename = f"{section_id}_Table1_FinalModel_Choice_and_GlobalMetrics.csv"
table_path = os.path.join(METRICS_DIR, table_filename)
final_model_df.to_csv(table_path, index=False)

print(f"\n[INFO] Saved Section 10.1 Table 1 (final model choice and global metrics) to:")
print(table_path)
print(f"[INFO] Suggested APA label: Table X. {TABLE_TITLE}.")

display(final_model_df)

In [ ]:
# --- 10.2. Helper: evaluate metrics within a subgroup with APA-style exports ---

section_id  = "Section10_2"
TABLE_TITLE = "Subgroup-specific performance of the final model on the test set"

def subgroup_metrics(y_true, y_proba, y_pred):
    """
    Compute key metrics for a subgroup.
    """
    if len(np.unique(y_true)) < 2:
        # AUROC undefined if only one class present; skip metrics
        return {
            "n": len(y_true),
            "AUROC": np.nan,
            "AUPRC": np.nan,
            "Accuracy": np.nan,
            "Precision": np.nan,
            "Recall": np.nan,
            "F1": np.nan
        }
    
    auc = roc_auc_score(y_true, y_proba)
    ap = average_precision_score(y_true, y_proba)
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    
    return {
        "n": len(y_true),
        "AUROC": auc,
        "AUPRC": ap,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1
    }

def evaluate_subgroups(X_test, y_test, y_proba, y_pred, group_col: str):
    """
    Evaluate model performance within each category of group_col.
    Returns a DataFrame indexed by group_col.
    """
    subgroup_results = []
    for level in sorted(X_test[group_col].unique()):
        mask = X_test[group_col] == level
        y_true_group  = y_test[mask]
        y_proba_group = y_proba[mask]
        y_pred_group  = y_pred[mask]
        
        metrics_dict = subgroup_metrics(y_true_group, y_proba_group, y_pred_group)
        metrics_dict[group_col] = level
        subgroup_results.append(metrics_dict)
    
    subgroup_df = pd.DataFrame(subgroup_results).set_index(group_col)
    return subgroup_df

# -------------------------------------------------------------------------
# Loop over subgroup-defining variables, print, save tables & plots
# -------------------------------------------------------------------------
group_cols = ["Sex", "AgeCategory", "Race", "Diabetic"]

for i, group_col in enumerate(group_cols, start=1):
    print(f"\n=== Subgroup performance by {group_col} ({final_model_name}) ===")
    
    subgroup_df = evaluate_subgroups(
        X_test,
        y_test,
        y_test_proba_final,
        y_test_pred_final,
        group_col=group_col
    )
    
    # Sort by AUROC (descending) for nicer display
    subgroup_sorted = subgroup_df.sort_values("AUROC", ascending=False)
    display(subgroup_sorted)
    
    # -------------------- Save APA-style table ---------------------------
    export_df = subgroup_sorted.reset_index()
    
    # Round numeric columns for readability
    for col in ["AUROC", "AUPRC", "Accuracy", "Precision", "Recall", "F1"]:
        export_df[col] = export_df[col].round(4)
    
    table_filename = f"{section_id}_Table{i}_SubgroupMetrics_by{group_col}.csv"
    table_path = os.path.join(METRICS_DIR, table_filename)
    export_df.to_csv(table_path, index=False)
    
    print(f"[INFO] Saved {section_id} Table {i} (subgroup metrics by {group_col}) to:")
    print(table_path)
    print(
        f"[INFO] Suggested APA label: Table X. "
        f"Test-set performance of {final_model_name} by {group_col} subgroup."
    )
    
    # -------------------- Save AUROC barplot per subgroup ----------------
    plt.figure(figsize=(8, 5))
    sns.barplot(
        data=export_df,
        x="AUROC",
        y=group_col
    )
    plt.xlabel("Area under the ROC curve (AUROC)")
    plt.ylabel(group_col)
    plt.title(f"Test-set AUROC by {group_col} subgroup – {final_model_name}")
    plt.tight_layout()
    
    plot_filename = f"{section_id}_Plot{i}_AUROC_by{group_col}.png"
    plot_path = os.path.join(PLOTS_DIR, plot_filename)
    plt.savefig(plot_path, dpi=300, bbox_inches="tight")
    
    print(f"[INFO] Saved {section_id} Plot {i} (AUROC by {group_col} subgroup) to:")
    print(plot_path)
    
    plt.show()

## 11. How This Notebook Supports the Research Questions

**RQ1 / H1 (Risk Factors)** – Are traditional and lifestyle risk factors strongly
associated with self-reported heart disease, and do they rank as top predictors?

Evidence sources in this notebook:

- Prevalence plots by **AgeCategory**, **GenHealth**, **Smoking**, **Diabetic**, **PhysicalActivity**, etc.
- Multivariable logistic regression (statsmodels) with odds ratios and p-values.
- Feature importance from Random Forest and XGBoost.
- SHAP global importance and effect plots.

**RQ2 / H2 (Model Performance)** – Do non-linear ML models outperform logistic regression?

Evidence sources in this notebook:

- Cross-validated AUROC for tuned Logistic Regression, Random Forest, XGBoost.
- Paired t-tests comparing per-fold AUROCs (non-linear vs logistic).
- Test-set metrics (AUROC, AUPRC, sensitivity, etc.) for all models.

These outputs (tables, metrics, plots) can be exported or copied into your
manuscript to justify acceptance or rejection of each hypothesis.